# Resumen Sachs: Beta=0 vs Beta óptima (kan y kaam)

Analiza, para los modelos `kan` y `kaam` sobre el caso semi-sintético **Sachs**
(`notebooks/Experimento3 sachs/sachs_hsic_beta.ipynb`), si la mejora del modelo con término HSIC
(`kan_hsic`/`kaam_hsic`, `loss='hybrid'` con la `Beta` óptima) frente al modelo base (`Beta=0`,
`loss='mse'`) es significativa, con un test de Wilcoxon pareado -- igual que en
`Resumen_Observacional.ipynb` / `Resumen_Intervencional.ipynb` / `Resumen_Sinteticos_*.ipynb`.

**Requisito de orden**: este notebook lee los pickles que genera `sachs_hsic_beta.ipynb`
(`outputs/sachs_hsic/data/sachs_{beta_sweep,beta0_sweep,results}_*.pkl`), y recalcula aquí la misma
Beta óptima que esa selecciona (mismo criterio) solo para dejarla explícita en las tablas -- si se
ejecuta este notebook con pickles de una ejecución anterior de `sachs_hsic_beta.ipynb` que usara otro
criterio de selección (en particular, `sachs_hsic_beta.ipynb` sigue seleccionando por `min(rf_acc_obs)`
a secas, sin desempate explícito), la Beta mostrada aquí podría no coincidir con la que realmente se
entrenó en `kan_hsic`/`kaam_hsic` si hubiera un empate exacto en `rf_acc_obs` -- en la práctica esto no
ocurre, ya que ambos comparten el mismo criterio primario.

**Cambios frente a la versión anterior de este notebook**:

- `sachs_hsic_beta.ipynb` ahora corre un barrido de 20 semillas (`beta_sweep_seeds`, 42-61) tanto para
  el baseline `Beta=0` como para cada `Beta` del barrido, por familia (`kan`, `kaam`), y también para
  `dbcm`/`flow`. Eso da un valor de `rf_acc_obs`/`mmd_obs` por semilla
  (`sachs_beta0_sweep_<familia>_por_semilla.pkl`, `sachs_beta_sweep_<familia>_por_semilla.pkl`,
  `sachs_{dbcm,flow}_sweep_por_semilla.pkl`), lo que permite aplicar Wilcoxon pareado (por semilla)
  también a las métricas observacionales -- antes solo existía un valor agregado único por modelo
  (Sachs se generaba con `seed=42` fijo) y esas métricas se mostraban sin test de significancia.
- La selección de la Beta óptima usa ahora el mismo criterio de prioridad estricta que
  `Resumen_Observacional.ipynb` / `Resumen_Intervencional.ipynb` / `Resumen_Sinteticos_*.ipynb`
  (commit `4e6a274`): minimizar `RF Acc` -> desempate por `MMD` -> desempate por la `Beta` más pequeña.
  A diferencia de esos notebooks, aquí **no hay un nivel de desempate por "MAE Z"**: el barrido de
  Sachs (`sachs_beta_sweep_<familia>_por_semilla.pkl`) solo guarda `mmd_obs`/`rf_acc_obs`/
  `training_time` por semilla -- no existe una métrica tipo "MAE Z" para Sachs sin volver a entrenar el
  barrido completo en `sachs_hsic_beta.ipynb`.

**Salidas de este notebook**: exactamente dos tablas (más abajo), cada una con una columna de p-valor
junto a cada métrica. En **negrita** las celdas (valor y p-valor) donde la mejora de la Beta óptima
frente a `Beta=0` es significativa (p < 0.05, Wilcoxon unidireccional `alternative='less'`, H1 = la
Beta óptima es menor/mejor que `Beta=0`). El p-valor solo aplica a las filas `... (opt)` /
`kan_hsic`/`kaam_hsic` (comparadas frente a su baseline `... (mse)` / `kan`/`kaam` de la misma
familia); `DBCM`/`Causal flow` no participan en esta comparación y se muestran sin p-valor (`-`).


In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

NOTEBOOKS_DIR = Path.cwd().parent if Path.cwd().name == "Resumenes" else Path.cwd()
REPO_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = REPO_ROOT / "outputs" / "sachs_hsic" / "data"
TABLAS_DIR = NOTEBOOKS_DIR / "Resumenes" / "tablas"

ALPHA = 0.05
FAMILIAS = ["kan", "kaam"]
BETAS = [round(0.1 * i, 1) for i in range(1, 9)]  # 0.1 .. 0.8, igual que en sachs_hsic_beta.ipynb
MODEL_NAMES = ["kan", "kaam", "kan_hsic", "kaam_hsic", "dbcm", "flow"]

## Carga de resultados (`outputs/sachs_hsic/data`)

Por familia (`kan`, `kaam`): `sachs_beta_sweep_<familia>.pkl` (barrido agregado, media sobre 20
semillas, por Beta), `sachs_beta_sweep_<familia>_por_semilla.pkl` (barrido por semilla, por Beta) y
`sachs_beta0_sweep_<familia>_por_semilla.pkl` (baseline `Beta=0` por semilla). Para `dbcm`/`flow`, su
barrido de 20 semillas (`sachs_{dbcm,flow}_sweep_por_semilla.pkl`). Y, para los 6 modelos finales
(`MODEL_NAMES`), `sachs_results_<modelo>.pkl` (métricas agregadas `*_avg` y, para las interventional/
counterfactual, listas `*_all` con un valor por cada uno de los 660 escenarios de intervención).


In [2]:
def cargar_pickle(nombre: str):
    path = DATA_DIR / f"{nombre}.pkl"
    assert path.exists(), f"No existe: {path}"
    with open(path, "rb") as f:
        return pickle.load(f)


beta_sweep_agg = {familia: cargar_pickle(f"sachs_beta_sweep_{familia}") for familia in FAMILIAS}
beta_sweep_seed = {familia: cargar_pickle(f"sachs_beta_sweep_{familia}_por_semilla") for familia in FAMILIAS}
beta0_seed = {familia: cargar_pickle(f"sachs_beta0_sweep_{familia}_por_semilla") for familia in FAMILIAS}

dbcm_seed = cargar_pickle("sachs_dbcm_sweep_por_semilla")
flow_seed = cargar_pickle("sachs_flow_sweep_por_semilla")

resultados = {modelo: cargar_pickle(f"sachs_results_{modelo}") for modelo in MODEL_NAMES}


## Selección de la Beta óptima

Criterio de prioridad estricta (igual que `Resumen_Observacional.ipynb`, sin el nivel de "MAE Z" --
ver introducción): minimizar `rf_acc_obs` -> desempate por `mmd_obs` -> desempate por la `Beta` más
pequeña. `Beta=0` nunca es candidata (no está en `sachs_beta_sweep_<familia>.pkl`).


In [3]:
def seed_values(per_seed_dict: dict, metric: str):
    '''Devuelve la lista de valores de `metric` para todas las semillas en `per_seed_dict`.'''
    return [v[metric] for v in per_seed_dict.values()]


def beta_optima_criterio(sweep_agg_familia: dict):
    '''Selecciona la Beta optima (Beta != 0) por prioridad estricta: RF Acc -> MMD -> Beta mas pequena.

    `sweep_agg_familia` es sachs_beta_sweep_<familia>.pkl: {beta: {"mmd_obs", "rf_acc_obs", ...}}, ya
    promediado sobre las 20 semillas del barrido (sachs_hsic_beta.ipynb). A diferencia de
    Resumen_Observacional.ipynb / Resumen_Sinteticos_*.ipynb, no hay un tercer nivel de desempate por
    "MAE Z": Sachs no guarda esa metrica en el barrido.

    Devuelve (beta_opt: float, candidatas: pd.DataFrame, criterio: str).
    '''
    candidatas = pd.DataFrame(sweep_agg_familia).T[["rf_acc_obs", "mmd_obs"]]
    candidatas.index.name = "Beta"
    candidatas = candidatas.reset_index().sort_values(
        by=["rf_acc_obs", "mmd_obs", "Beta"],
    ).reset_index(drop=True)

    ganadora = candidatas.iloc[0]
    beta_opt = float(ganadora["Beta"])

    eps = 1e-9
    empatadas_rf = candidatas[abs(candidatas["rf_acc_obs"] - ganadora["rf_acc_obs"]) < eps]
    if len(empatadas_rf) == 1:
        criterio = "menor rf_acc_obs"
    else:
        empatadas_mmd = empatadas_rf[abs(empatadas_rf["mmd_obs"] - ganadora["mmd_obs"]) < eps]
        if len(empatadas_mmd) == 1:
            criterio = f"empate en rf_acc_obs entre {len(empatadas_rf)} betas, desempate por menor mmd_obs"
        else:
            criterio = f"empate en rf_acc_obs y mmd_obs entre {len(empatadas_mmd)} betas, desempate por la Beta mas pequena"

    return beta_opt, candidatas, criterio


beta_optima = {}
for familia in FAMILIAS:
    beta_optima[familia], candidatas, criterio = beta_optima_criterio(beta_sweep_agg[familia])
    print(f"Beta optima ({familia}): {beta_optima[familia]}  [{criterio}]")


Beta optima (kan): 0.2  [menor rf_acc_obs]
Beta optima (kaam): 0.3  [menor rf_acc_obs]


## Test de Wilcoxon pareado (Beta óptima vs Beta=0)

Dos comparaciones pareadas, unidireccionales (`alternative='less'`; H1: la Beta óptima es menor/mejor
que `Beta=0`):

- **Observacional** (`rf_acc_obs`, `mmd_obs`): pareado por semilla (20 semillas, `beta_sweep_seeds` en
  `sachs_hsic_beta.ipynb`) entre el baseline `Beta=0` y la Beta óptima, por familia.
- **Interventional / counterfactual** (`rf_acc_int`, `mmd_int`, `mae_cf`): pareado por escenario de
  intervención (660 pares nodo x valor, `inter_vector`) entre el modelo base (`kan`/`kaam`) y su
  versión HSIC (`kan_hsic`/`kaam_hsic`).


In [4]:
def wilcoxon_menor(a_vals, b_vals) -> float:
    '''Wilcoxon signed-rank pareado. H0: sin diferencia. H1 (alternative='less'): b es menor (mejor) que a.

    Devuelve NaN si el test no esta definido (p.ej. todas las diferencias son nulas).
    '''
    a_vals = np.asarray(a_vals, dtype=float)
    b_vals = np.asarray(b_vals, dtype=float)
    assert len(a_vals) == len(b_vals), "Longitudes distintas"
    try:
        _, p = wilcoxon(b_vals, a_vals, alternative="less")
    except ValueError:
        p = float("nan")
    return p


METRICAS_OBS = ["rf_acc_obs", "mmd_obs"]
METRICAS_INT = ["rf_acc_int", "mmd_int", "mae_cf"]

p_obs = {}
for familia in FAMILIAS:
    p_obs[familia] = {}
    for metrica in METRICAS_OBS:
        a_vals = seed_values(beta0_seed[familia], metrica)
        b_vals = seed_values(beta_sweep_seed[familia][beta_optima[familia]], metrica)
        p_obs[familia][metrica] = wilcoxon_menor(a_vals, b_vals)

p_int = {}
for familia in FAMILIAS:
    p_int[familia] = {}
    base = resultados[familia]
    hsic = resultados[f"{familia}_hsic"]
    for metrica in METRICAS_INT:
        p_int[familia][metrica] = wilcoxon_menor(base[f"{metrica}_all"], hsic[f"{metrica}_all"])


## Función de formato: negrita en valor + p-valor cuando p < 0.05


In [5]:
def tabla_con_negrita(df: pd.DataFrame, columnas_valor_a_pvalor: dict):
    '''Muestra `df` con, por fila, la celda de valor y su p-valor emparejado en negrita cuando p < ALPHA.

    `columnas_valor_a_pvalor`: {columna_valor: columna_p_valor}. Filas sin p-valor (NaN, sin
    comparacion aplicable) no se resaltan y su p-valor se muestra como "-".
    '''
    columnas_p = set(columnas_valor_a_pvalor.values())

    def resaltar(row):
        estilos = []
        for col in row.index:
            en_negrita = False
            if col in columnas_valor_a_pvalor:
                p = row[columnas_valor_a_pvalor[col]]
                en_negrita = pd.notna(p) and p < ALPHA
            elif col in columnas_p:
                en_negrita = pd.notna(row[col]) and row[col] < ALPHA
            estilos.append("font-weight: bold" if en_negrita else "")
        return estilos

    return df.style.apply(resaltar, axis=1).format(
        {col: "{:.4f}".format for col in columnas_p}, na_rep="-",
    )


## Tabla 1: métricas observacionales (media +- std por modelo, con p-valor)

Mismo formato que la tabla "Summary table" de `sachs_hsic_beta_results.ipynb` (media +- std sobre las
20 semillas de cada caja), con una columna de p-valor junto a `rf_acc_obs` y otra junto a `mmd_obs`. El
p-valor solo aplica a las filas `... (opt)` (comparadas frente a su baseline `... (mse)` de la misma
familia); en negrita cuando p < 0.05.


In [6]:
BOX_MODELS = [
    (beta0_seed["kan"], "KAN (mse)", None),
    (beta_sweep_seed["kan"][beta_optima["kan"]], f"KAN beta={beta_optima['kan']:g} (opt)", "kan"),
    (beta0_seed["kaam"], "KAAM (mse)", None),
    (beta_sweep_seed["kaam"][beta_optima["kaam"]], f"KAAM beta={beta_optima['kaam']:g} (opt)", "kaam"),
    (dbcm_seed, "DBCM", None),
    (flow_seed, "Causal flow", None),
]

filas_obs = []
for per_seed, label, familia in BOX_MODELS:
    rf_vals = seed_values(per_seed, "rf_acc_obs")
    mmd_vals = seed_values(per_seed, "mmd_obs")
    p_rf = p_obs[familia]["rf_acc_obs"] if familia is not None else float("nan")
    p_mmd = p_obs[familia]["mmd_obs"] if familia is not None else float("nan")
    filas_obs.append({
        "model": label,
        "rf_acc_obs": f"{np.mean(rf_vals):.4f} +- {np.std(rf_vals):.4f}",
        "p_valor_rf_acc_obs": p_rf,
        "mmd_obs": f"{np.mean(mmd_vals):.4f} +- {np.std(mmd_vals):.4f}",
        "p_valor_mmd_obs": p_mmd,
    })

tabla_obs = pd.DataFrame(filas_obs)
tabla_con_negrita(tabla_obs, {"rf_acc_obs": "p_valor_rf_acc_obs", "mmd_obs": "p_valor_mmd_obs"})


,model,rf_acc_obs,p_valor_rf_acc_obs,mmd_obs,p_valor_mmd_obs
0,KAN (mse),0.5262 +- 0.0185,-,0.0071 +- 0.0016,-
1,KAN beta=0.2 (opt),0.5179 +- 0.0183,0.1179,0.0072 +- 0.0017,0.8695
2,KAAM (mse),0.5279 +- 0.0239,-,0.0071 +- 0.0016,-
3,KAAM beta=0.3 (opt),0.5217 +- 0.0193,0.2177,0.0072 +- 0.0017,0.8350
4,DBCM,0.5775 +- 0.0399,-,0.0085 +- 0.0025,-
5,Causal flow,0.5621 +- 0.0279,-,0.0066 +- 0.0016,-


## Tabla 2: métricas interventional / counterfactual (media +- std por modelo, con p-valor)

Mismo formato que la tabla `int_cf_table` de `sachs_hsic_beta_results.ipynb` (media +- std sobre los
660 escenarios de intervención), con una columna de p-valor junto a cada métrica. El p-valor solo
aplica a `kan_hsic`/`kaam_hsic` (comparados frente a `kan`/`kaam`); en negrita cuando p < 0.05.


In [7]:
filas_int = []
for modelo in MODEL_NAMES:
    familia = modelo[:-len("_hsic")] if modelo.endswith("_hsic") else None
    results = resultados[modelo]
    fila = {"model": modelo}
    for metrica in METRICAS_INT:
        values = results[f"{metrica}_all"]
        fila[metrica] = f"{np.mean(values):.4f} +- {np.std(values):.4f}"
        fila[f"p_valor_{metrica}"] = p_int[familia][metrica] if familia is not None else float("nan")
    filas_int.append(fila)

tabla_int = pd.DataFrame(filas_int)
tabla_con_negrita(tabla_int, {m: f"p_valor_{m}" for m in METRICAS_INT})


,model,rf_acc_int,p_valor_rf_acc_int,mmd_int,p_valor_mmd_int,mae_cf,p_valor_mae_cf
0,kan,0.5653 +- 0.0689,-,0.0197 +- 0.0523,-,0.0815 +- 0.1795,-
1,kaam,0.5687 +- 0.0701,-,0.0200 +- 0.0525,-,0.0834 +- 0.1813,-
2,kan_hsic,0.5731 +- 0.0658,1.0000,0.0197 +- 0.0531,0.0000,0.0824 +- 0.1823,1.0000
3,kaam_hsic,0.5732 +- 0.0695,1.0000,0.0199 +- 0.0533,0.0000,0.0844 +- 0.1839,1.0000
4,dbcm,0.5809 +- 0.0485,-,0.0115 +- 0.0072,-,0.0596 +- 0.0667,-
5,flow,0.5756 +- 0.0579,-,0.0103 +- 0.0138,-,0.0466 +- 0.0727,-


## Guardar tablas (CSV)


In [8]:
tablas_dir = TABLAS_DIR
tablas_dir.mkdir(parents=True, exist_ok=True)

tabla_obs.to_csv(tablas_dir / "resumen_sachs_obs.csv", index=False)
tabla_int.to_csv(tablas_dir / "resumen_sachs_int.csv", index=False)

print(f"Guardado en: {tablas_dir / 'resumen_sachs_obs.csv'}")
print(f"Guardado en: {tablas_dir / 'resumen_sachs_int.csv'}")

Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_sachs_obs.csv
Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_sachs_int.csv
